# Advanced Retrieval — Hands-On

Offline simulation of dense+lexical hybrid retrieval, RRF, reranking, filters, and MMR.

## 0. Setup

In [ ]:
%pip install -q numpy
import hashlib, numpy as np
corpus = {"a":"refund policy receipt thirty days", "b":"return item with SKU ABC123", "c":"shipping delay tracking number", "d":"refund exception manager approval", "e":"warranty repair device serial"}
metadata = {"a":{"product":"retail"},"b":{"product":"retail"},"c":{"product":"logistics"},"d":{"product":"retail"},"e":{"product":"hardware"}}
def toks(x): return x.lower().split()
def embed(text, dim=32):
    v=np.zeros(dim)
    for t in toks(text): v[int(hashlib.md5(t.encode()).hexdigest(),16)%dim]+=1
    return v/(np.linalg.norm(v)+1e-9)
vecs={k:embed(v) for k,v in corpus.items()}

## 1. Dense and lexical rankings

In [ ]:
def dense_rank(q):
    qv=embed(q); return sorted([(float(qv@v), k) for k,v in vecs.items()], reverse=True)
def lexical_rank(q):
    qt=set(toks(q)); return sorted([(len(qt & set(toks(txt))), k) for k,txt in corpus.items()], reverse=True)
q="refund for SKU ABC123"
print("dense", dense_rank(q))
print("lexical", lexical_rank(q))

## 2. RRF fusion

In [ ]:
def rrf_rank(*rankings, k=60):
    scores={}
    for ranking in rankings:
        for r, (_, doc) in enumerate(ranking, 1): scores[doc]=scores.get(doc,0)+1/(k+r)
    return sorted([(s,d) for d,s in scores.items()], reverse=True)
fused=rrf_rank(dense_rank(q), lexical_rank(q))
print(fused)

## 3. Metadata filtering before reranking

In [ ]:
filtered=[(s,d) for s,d in fused if metadata[d]["product"]=="retail"]
print(filtered)

## 4. Simulated cross-encoder reranker

In [ ]:
def rerank_score(q, doc):
    qt=set(toks(q)); dt=set(toks(corpus[doc])); return 2*len(qt & dt)+0.1*len(dt)
reranked=sorted([(rerank_score(q,d), d) for _,d in filtered], reverse=True)
print(reranked)

## 5. MMR final selection

In [ ]:
def mmr(candidates, k=2, lam=0.7):
    qv=embed(q); selected=[]; remaining=[d for _,d in candidates]
    while remaining and len(selected)<k:
        best=None; best_score=-9
        for d in remaining:
            red=max([float(vecs[d]@vecs[s]) for s in selected] or [0])
            score=lam*float(qv@vecs[d])-(1-lam)*red
            if score>best_score: best=d; best_score=score
        selected.append(best); remaining.remove(best)
    return selected
print(mmr(reranked, 2))

## 6. Exercise prompts
1. Change RRF k.
2. Add a date filter.
3. Lower MMR lambda and inspect diversity.